# 4. Bi-Temporal Knowledge Graph over 58 SOC Runbooks — Stale Knowledge, Conflicts and Agent Memory

**Presentation map:** Slide 38 (facts that change over time → temporal graph memory), Slide 49 (every assertion carries
time and status), Slide 52 (memory tiers: episodic → semantic), Slide 53 (bi-temporal edges; a contradiction
*invalidates* the old edge instead of deleting it), Slides 54-55 (temporal graph memory: better on time).

**Dataset:** `data/runbooks/documents.jsonl` — 58 SOC incident-response runbooks (29 English + 29 Farsi) with version,
status, publication date and supersession metadata — and `data/runbooks/questions.jsonl`, a 100-question benchmark
with gold runbooks, gold owners and gold next steps. Two traps are built into the data:

- **Stale:** `ransomware-contain-en-v1` (2023, *isolate first, no memory snapshot*) was superseded in May 2026 by
  `ransomware-contain-en-v2` (*snapshot, then isolate*). The two give contradictory instructions.
- **Conflict:** `cloud-owner-soc-en-v1` and `cloud-owner-platform-en-v1` are **both current** and disagree on who may
  lock a leaked IAM key. A correct system must *flag* the conflict, not silently pick one.

**What we measure:** the same embedding model, the same `k = 3`, the same answer prompt — the only difference is
whether retrieval goes through the bi-temporal graph. We evaluate on all 52 English answerable, stale and conflict
questions plus 4 probes phrased in the old runbook's wording.

In [1]:
import json, re
from collections import Counter
import networkx as nx

with open("data/runbooks/documents.jsonl") as f:
    docs = [json.loads(line) for line in f]
DOC = {d["doc_id"]: d for d in docs}
en_docs = [d for d in docs if d["language"] == "en"]
print(f"Loaded {len(docs)} runbooks ({len(en_docs)} English, {len(docs) - len(en_docs)} Farsi)")

G = nx.MultiDiGraph()
for d in docs:
    G.add_node(d["doc_id"], label="Runbook", **{k: v for k, v in d.items() if k not in ("sections", "body")})
    fam = ("IncidentType", d["incident_type"], d["language"])
    G.add_node(fam, label="IncidentType")
    # GOVERNED_BY carries the bi-temporal envelope: valid_at (event time) and invalid_at (filled in when superseded)
    G.add_edge(fam, d["doc_id"], key="GOVERNED_BY", type="GOVERNED_BY",
               version=d["version"], valid_at=d["published_at"], invalid_at=None)
    role = ("Role", d["owner_role"])
    G.add_node(role, label="Role")
    G.add_edge(d["doc_id"], role, key="OWNED_BY", type="OWNED_BY")

# SUPERSEDES: the newer runbook invalidates the older edge - the old node and edge stay queryable as history
for d in docs:
    if d.get("supersedes"):
        G.add_edge(d["doc_id"], d["supersedes"], key="SUPERSEDES", type="SUPERSEDES", at=d["published_at"])
        G.nodes[d["supersedes"]]["invalid_at"] = d["published_at"]
        for u, v, k, e in G.in_edges(d["supersedes"], keys=True, data=True):
            if k == "GOVERNED_BY":
                e["invalid_at"] = d["published_at"]

# CONFLICTS_WITH: two *current* runbooks on the same decision (same incident type + title stem) with different owners
def decision_key(d):
    return (d["incident_type"], d["language"], d["title"].split(" — ")[0])
by_key = {}
for d in docs:
    if d["status"] == "current":
        by_key.setdefault(decision_key(d), []).append(d)
for key, group in by_key.items():
    for a in group:
        for b in group:
            if a["doc_id"] < b["doc_id"] and a["owner_role"] != b["owner_role"]:
                G.add_edge(a["doc_id"], b["doc_id"], key="CONFLICTS_WITH", type="CONFLICTS_WITH", on=key[2])
                G.add_edge(b["doc_id"], a["doc_id"], key="CONFLICTS_WITH", type="CONFLICTS_WITH", on=key[2])

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("Edges by type:", dict(Counter(k for _, _, k in G.edges(keys=True))))
for u, v, k, e in G.edges(keys=True, data=True):
    if "-en-" in str(u) and (k == "SUPERSEDES" or (k == "CONFLICTS_WITH" and u < v)):
        print(f"  ({u})-[:{k}]->({v})  {e}")

Loaded 58 runbooks (29 English, 29 Farsi)
Graph: 92 nodes, 122 edges
Edges by type: {'OWNED_BY': 58, 'GOVERNED_BY': 58, 'SUPERSEDES': 2, 'CONFLICTS_WITH': 4}
  (ransomware-contain-en-v2)-[:SUPERSEDES]->(ransomware-contain-en-v1)  {'type': 'SUPERSEDES', 'at': '2026-05-10'}
  (cloud-owner-platform-en-v1)-[:CONFLICTS_WITH]->(cloud-owner-soc-en-v1)  {'type': 'CONFLICTS_WITH', 'on': 'Cloud IAM lock ownership'}


## Two retrievers, one difference (Slides 38, 53)

- **Naive vector RAG:** top-3 runbooks by cosine similarity over `title + body`. No notion of time or status.
- **Bi-temporal graph RAG:** the *same* similarity ranking, but candidates come from the graph: only runbooks whose
  `GOVERNED_BY` edge is valid at `as_of`, every hit is expanded along `CONFLICTS_WITH`, and each passage is labelled
  with its version, validity and any conflict, so the LLM sees the graph's metadata, not just text.

In [2]:
import os
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
EMBED_MODEL, CHAT_MODEL, K = "text-embedding-3-small", "gpt-4o-mini", 3

def embed(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    m = np.array([e.embedding for e in resp.data])
    return m / np.linalg.norm(m, axis=1, keepdims=True)

def chat(system, user):
    return client.chat.completions.create(model=CHAT_MODEL, temperature=0, seed=7, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": user}]).choices[0].message.content

doc_matrix = embed([f"{d['title']}\n{d['body']}" for d in en_docs])
TODAY = "2026-10-17"

def valid_at(doc_id, as_of):
    e = G.edges[("IncidentType", DOC[doc_id]["incident_type"], DOC[doc_id]["language"]), doc_id, "GOVERNED_BY"]
    return e["valid_at"] <= as_of and (e["invalid_at"] is None or as_of < e["invalid_at"])

def naive_retrieve(qvec, k=K):
    order = np.argsort(-(doc_matrix @ qvec))
    return [en_docs[i]["doc_id"] for i in order[:k]]

def graph_retrieve(qvec, k=K, as_of=TODAY):
    order = np.argsort(-(doc_matrix @ qvec))
    hits = [en_docs[i]["doc_id"] for i in order if valid_at(en_docs[i]["doc_id"], as_of)][:k]
    for h in list(hits):                                     # expand along CONFLICTS_WITH
        for _, v, key in G.out_edges(h, keys=True):
            if key == "CONFLICTS_WITH" and v not in hits:
                hits.append(v)
    return hits

def context_block(doc_ids, with_metadata):
    parts = []
    for did in doc_ids:
        head = f"[{did}]"
        if with_metadata:
            n = G.nodes[did]
            conflicts = [v for _, v, key in G.out_edges(did, keys=True) if key == "CONFLICTS_WITH"]
            head += (f" version={n['version']} valid_from={n['published_at']} status={n['status']}"
                     + (f" CONFLICTS_WITH={conflicts} (both current; owners disagree)" if conflicts else ""))
        parts.append(f"{head}\n{DOC[did]['body']}")
    return "\n\n".join(parts)

ANSWER_SYSTEM = ("You are a SOC assistant. Answer the analyst's question in 2-3 sentences using only the runbook context. "
                 "Cite runbook ids.")

def answer(question, doc_ids, with_metadata):
    return chat(ANSWER_SYSTEM, f"Question: {question}\n\nRunbook context:\n{context_block(doc_ids, with_metadata)}")

# Benchmark: every English answerable / stale / conflict question + 4 probes phrased in the OLD runbook's wording
with open("data/runbooks/questions.jsonl") as f:
    questions = [json.loads(l) for l in f]
bench = [q for q in questions if q["language"] == "en" and q["kind"] in ("answerable", "stale", "conflict")]
PROBES = ["Should we immediately isolate the ransomware host without taking a memory snapshot?",
          "Is it right to pull the network cable first on an encrypting host and skip imaging?",
          "Our ransomware runbook says isolate-first. What exactly do we do on a reachable host?",
          "Can we delay isolation of a ransomware host to capture memory?"]
for i, text in enumerate(PROBES, 1):
    bench.append({"question_id": f"probe-old-wording-{i}", "kind": "stale-probe", "text": text,
                  "gold_doc_ids": ["ransomware-contain-en-v2"], "gold_owner": None,
                  "gold_next_step_contains": "memory snapshot first, then isolate (if EDR reachable)"})
print("Benchmark:", dict(Counter(q["kind"] for q in bench)), f"= {len(bench)} questions")

q_matrix = embed([q["text"] for q in bench])
for q, qv in zip(bench, q_matrix):
    q["naive_docs"], q["graph_docs"] = naive_retrieve(qv), graph_retrieve(qv)

Benchmark: {'answerable': 45, 'conflict': 3, 'stale': 4, 'stale-probe': 4} = 56 questions


## Answer and judge

Both pipelines generate an answer with the same prompt. A strict LLM judge sees the **gold runbook text**, the gold
next step / owner, and the question kind. For conflict questions the only correct answer is one that **flags that
two current runbooks disagree** instead of picking one owner.

In [3]:
from pydantic import BaseModel

class Verdict(BaseModel):
    correct: bool
    reason: str

def judge(q, ans):
    gold_text = "\n\n".join(DOC[g]["body"] for g in q["gold_doc_ids"])
    rule = ("This is a CONFLICT question: two current runbooks disagree. The answer is correct ONLY if it explicitly says the "
            "runbooks conflict / disagree (or names both owners as competing) instead of presenting one owner as settled."
            if q["kind"] == "conflict" else
            "The answer is correct only if it agrees with the GOLD runbook and does not recommend anything the gold runbook "
            "forbids (e.g. superseded instructions).")
    msg = (f"Question: {q['text']}\nGold next step contains: {q.get('gold_next_step_contains')}\nGold owner: {q.get('gold_owner')}\n"
           f"GOLD runbook text:\n{gold_text}\n\nCandidate answer:\n{ans}\n\n{rule}")
    return client.beta.chat.completions.parse(model=CHAT_MODEL, temperature=0, seed=7, response_format=Verdict, messages=[
        {"role": "system", "content": "You are a strict grader of SOC runbook answers."},
        {"role": "user", "content": msg}]).choices[0].message.parsed

def evaluate(q):
    out = {}
    for name, ids, meta in (("naive", q["naive_docs"], False), ("graph", q["graph_docs"], True)):
        ans = answer(q["text"], ids, meta)
        out[name] = {"answer": ans, "correct": judge(q, ans).correct,
                     "recall": all(g in ids for g in q["gold_doc_ids"]),
                     "stale": any(DOC[d]["status"] == "superseded" for d in ids),
                     "top1_stale": DOC[ids[0]]["status"] == "superseded"}
    return out

with ThreadPoolExecutor(max_workers=8) as pool:
    for q, res in zip(bench, pool.map(evaluate, bench)):
        q.update(res)

def rate(kind_filter, system, metric):
    sel = [q for q in bench if kind_filter(q)]
    return sum(q[system][metric] for q in sel) / len(sel), len(sel)

groups = [("all", lambda q: True), ("answerable", lambda q: q["kind"] == "answerable"),
          ("stale (+probes)", lambda q: q["kind"] in ("stale", "stale-probe")), ("conflict", lambda q: q["kind"] == "conflict")]
print(f"{'questions':<17}{'n':>4} | {'recall@3':>9} {'stale ctx':>10} {'correct':>8} | {'recall@3':>9} {'stale ctx':>10} {'correct':>8}")
print(f"{'':<22}|{'naive vector RAG':^30}|{'bi-temporal graph RAG':^30}")
for label, f in groups:
    row = f"{label:<17}{rate(f, 'naive', 'correct')[1]:>4} |"
    for system in ("naive", "graph"):
        row += f" {rate(f, system, 'recall')[0]:>9.0%} {rate(f, system, 'stale')[0]:>10.0%} {rate(f, system, 'correct')[0]:>8.0%} |"
    print(row)

questions           n |  recall@3  stale ctx  correct |  recall@3  stale ctx  correct
                      |       naive vector RAG       |    bi-temporal graph RAG     
all                56 |       95%        16%      88% |       96%         0%      95% |
answerable         45 |       98%         4%      96% |       98%         0%      96% |
stale (+probes)     8 |       75%        88%      62% |       88%         0%      88% |
conflict            3 |      100%         0%      33% |      100%         0%     100% |


In [4]:
# Inspect where the two pipelines disagree: stale contamination and conflicts
for q in bench:
    if q["kind"] in ("stale", "stale-probe", "conflict") and q["naive"]["correct"] != q["graph"]["correct"]:
        print(f"[{q['question_id']}] {q['text']}")
        print(f"  naive docs {q['naive_docs']}  correct={q['naive']['correct']}\n    -> {q['naive']['answer'][:230]}")
        print(f"  graph docs {q['graph_docs']}  correct={q['graph']['correct']}\n    -> {q['graph']['answer'][:230]}\n")

[cloud-cf-en-01] Who owns locking the IAM user after an access key is found in a ticket?
  naive docs ['cloud-owner-soc-en-v1', 'cloud-owner-platform-en-v1', 'ato-owner-en-v1']  correct=False
    -> According to the runbook [cloud-owner-soc-en-v1], the SOC Lead owns the responsibility for locking the IAM user and disabling the access key when it is found in a ticket or repository. However, there is a conflict with the Cloud P
  graph docs ['cloud-owner-soc-en-v1', 'cloud-owner-platform-en-v1', 'ato-owner-en-v1']  correct=True
    -> The ownership of locking the IAM user after an access key is found in a ticket is disputed between the SOC Lead and the Cloud Platform Lead. According to the SOC runbook [cloud-owner-soc-en-v1], the SOC Lead is responsible for dis

[cloud-cf-en-02] May SOC disable the leaked access key without Cloud Platform?
  naive docs ['cloud-owner-soc-en-v1', 'cloud-owner-platform-en-v1', 'cloud-contain-en-v1']  correct=False
    -> No, the SOC cannot disable the leake

## Time travel: the same graph answers "what was the policy then?" (Slide 53)

Because the superseded runbook was *invalidated*, not deleted, a compliance audit can ask the graph for the policy in
force on any date — the same `valid_at` filter, a different `as_of`.

In [5]:
probe = "For a reachable ransomware host, should we take a memory snapshot before isolation?"
pv = embed([probe])[0]
for as_of in ["2024-06-01", TODAY]:
    ids = graph_retrieve(pv, k=1, as_of=as_of)
    n = G.nodes[ids[0]]
    print(f"as_of={as_of} -> {ids[0]} (v{n['version']}, valid_from={n['published_at']}, invalid_at={n.get('invalid_at')})")
    print("   ", chat(f"You answer a point-in-time compliance audit as of {as_of}. Use only the runbook text. One sentence.",
                     f"{probe}\n\n{context_block(ids, True)}"), "\n")

as_of=2024-06-01 -> ransomware-contain-en-v1 (v1.0, valid_from=2023-08-18, invalid_at=2026-05-10)


    No, you should not take a memory snapshot before isolation; immediately isolate the host from the network. 

as_of=2026-10-17 -> ransomware-contain-en-v2 (v2.0, valid_from=2026-05-10, invalid_at=None)


    Yes, take a memory snapshot first before isolating the ransomware host if EDR is still reachable. 



## Agent memory: episodes in, bi-temporal facts out (Slides 52-53)

The same mechanism is how a graph memory (Graphiti / Zep) works for an agent. Each conversation turn is stored as an
**episode** (lossless provenance); an LLM with a **Pydantic schema** extracts **facts**; when a new fact contradicts
an existing one for a single-valued relation (`WORKS_AT`, `REPORTS_TO`), the old edge gets `invalid_at` and
`expired_at` — it is never deleted, so "where did she work in 2024?" still has an answer.

In [6]:
from typing import Literal, Optional

class Fact(BaseModel):
    subject: str
    predicate: Literal["WORKS_AT", "REPORTS_TO", "LEADS_PROJECT", "ON_CALL_FOR"]
    object: str
    valid_from: Optional[str]   # ISO date when the fact started being true, if stated or implied

class Extraction(BaseModel):
    facts: list[Fact]

EPISODES = [  # (episode id, system time the agent received it, text)
    ("ep-12", "2024-03-05", "Hi, I'm Ana. I just joined Lab A as an incident-response engineer."),
    ("ep-31", "2024-09-12", "Ana here - I'm now leading the ransomware-playbook project, and I report to Marko."),
    ("ep-87", "2026-02-10", "Quick update from Ana: since January I am at Lab B, and my manager there is Elena."),
    ("ep-90", "2026-02-11", "Ana: I'm on call for cloud-exposure incidents this quarter."),
]
SINGLE_VALUED = {"WORKS_AT", "REPORTS_TO"}
M = nx.MultiDiGraph()   # the memory graph

for ep_id, received_at, text in EPISODES:
    M.add_node(ep_id, label="Episode", text=text, received_at=received_at)
    facts = client.beta.chat.completions.parse(model=CHAT_MODEL, temperature=0, seed=7, response_format=Extraction, messages=[
        {"role": "system", "content": f"Extract durable facts about people. Today is {received_at}. Resolve relative dates "
                                      "('since January') to ISO dates; if no date is implied use today's date. Use the person's first name as id."},
        {"role": "user", "content": text}]).choices[0].message.parsed.facts
    for f in facts:
        start = f.valid_from or received_at
        if f.predicate in SINGLE_VALUED:          # contradiction -> invalidate, never delete
            for u, v, k, e in list(M.out_edges(f.subject, keys=True, data=True)):
                if e["predicate"] == f.predicate and v != f.object and e["invalid_at"] is None:
                    e["invalid_at"], e["expired_at"] = start, received_at
                    print(f"  {ep_id}: INVALIDATED ({u})-[:{f.predicate}]->({v})  invalid_at={start}")
        M.add_edge(f.subject, f.object, key=f"{f.predicate}:{ep_id}", predicate=f.predicate, valid_at=start,
                   invalid_at=None, created_at=received_at, expired_at=None, episode=ep_id)
        print(f"  {ep_id}: ADDED ({f.subject})-[:{f.predicate}]->({f.object})  valid_at={start}")

def recall(subject, predicate, as_of):
    return [v for _, v, e in M.out_edges(subject, data=True) if e["predicate"] == predicate
            and e["valid_at"] <= as_of and (e["invalid_at"] is None or as_of < e["invalid_at"])]

print("\nMemory edges kept (history included):", M.number_of_edges())
for as_of in ["2024-06-01", "2025-06-01", TODAY]:
    print(f"as_of {as_of}: Ana WORKS_AT {recall('Ana', 'WORKS_AT', as_of)}, REPORTS_TO {recall('Ana', 'REPORTS_TO', as_of)}")

  ep-12: ADDED (Ana)-[:WORKS_AT]->(Lab A)  valid_at=2024-03-05


  ep-31: ADDED (Ana)-[:LEADS_PROJECT]->(ransomware-playbook project)  valid_at=2024-09-12
  ep-31: ADDED (Ana)-[:REPORTS_TO]->(Marko)  valid_at=2024-09-12


  ep-87: INVALIDATED (Ana)-[:WORKS_AT]->(Lab A)  invalid_at=2026-01-01
  ep-87: ADDED (Ana)-[:WORKS_AT]->(Lab B)  valid_at=2026-01-01
  ep-87: INVALIDATED (Ana)-[:REPORTS_TO]->(Marko)  invalid_at=2026-01-01
  ep-87: ADDED (Ana)-[:REPORTS_TO]->(Elena)  valid_at=2026-01-01


  ep-90: ADDED (Ana)-[:ON_CALL_FOR]->(cloud-exposure incidents)  valid_at=2026-02-11

Memory edges kept (history included): 6
as_of 2024-06-01: Ana WORKS_AT ['Lab A'], REPORTS_TO []
as_of 2025-06-01: Ana WORKS_AT ['Lab A'], REPORTS_TO ['Marko']
as_of 2026-10-17: Ana WORKS_AT ['Lab B'], REPORTS_TO ['Elena']


In [7]:
def pct(system, metric, f=lambda q: True):
    return f"{rate(f, system, metric)[0]:.0%}"
stale_f = lambda q: q["kind"] in ("stale", "stale-probe")
conf_f = lambda q: q["kind"] == "conflict"
ans_f = lambda q: q["kind"] == "answerable"

print("=" * 82)
print(f"RESULTS SUMMARY — Notebook 4: bi-temporal graph vs naive vector RAG ({len(bench)} English questions, k={K})")
print("=" * 82)
print(f"{'':44}{'naive vector':>14}{'bi-temporal graph':>20}")
print(f"{'Answer correct (all questions, LLM judge)':44}{pct('naive','correct'):>14}{pct('graph','correct'):>20}")
print(f"{'Answer correct - answerable':44}{pct('naive','correct',ans_f):>14}{pct('graph','correct',ans_f):>20}")
print(f"{'Answer correct - stale + old-wording probes':44}{pct('naive','correct',stale_f):>14}{pct('graph','correct',stale_f):>20}")
print(f"{'Conflict flagged (cloud IAM ownership)':44}{pct('naive','correct',conf_f):>14}{pct('graph','correct',conf_f):>20}")
print(f"{'Superseded runbook in context - stale qs':44}{pct('naive','stale',stale_f):>14}{pct('graph','stale',stale_f):>20}")
print(f"{'Superseded runbook ranked #1 - stale qs':44}{pct('naive','top1_stale',stale_f):>14}{pct('graph','top1_stale',stale_f):>20}")
print(f"{'Gold recall@3 (all questions)':44}{pct('naive','recall'):>14}{pct('graph','recall'):>20}")
print(f"\nTime travel: as_of 2024-06-01 -> ransomware-contain-en-v1, as_of {TODAY} -> ransomware-contain-en-v2 (same graph)")
print(f"Agent memory: {M.number_of_edges()} edges kept after contradictions; Ana WORKS_AT {recall('Ana','WORKS_AT','2024-06-01')} "
      f"in 2024 and {recall('Ana','WORKS_AT',TODAY)} now")
print("\nLecture message (Slides 38, 53-55): time and status belong in the graph, not in the prompt - invalidate,")
print("don't delete; filter by validity before ranking; surface conflicts instead of letting similarity pick one.")

RESULTS SUMMARY — Notebook 4: bi-temporal graph vs naive vector RAG (56 English questions, k=3)
                                              naive vector   bi-temporal graph
Answer correct (all questions, LLM judge)              88%                 95%
Answer correct - answerable                            96%                 96%
Answer correct - stale + old-wording probes            62%                 88%
Conflict flagged (cloud IAM ownership)                 33%                100%
Superseded runbook in context - stale qs               88%                  0%
Superseded runbook ranked #1 - stale qs                38%                  0%
Gold recall@3 (all questions)                          95%                 96%

Time travel: as_of 2024-06-01 -> ransomware-contain-en-v1, as_of 2026-10-17 -> ransomware-contain-en-v2 (same graph)
Agent memory: 6 edges kept after contradictions; Ana WORKS_AT ['Lab A'] in 2024 and ['Lab B'] now

Lecture message (Slides 38, 53-55): time and status bel